## Ollama Inferenz Server 

Beispiel für einen Client, der mit einem Ollama-Inference-Server auf einem separaten Rechner kommuniziert.

Auf diesem Server läuft der Ollama-Inference-Service als zentral betriebener Dienst und stellt das geladene Modell über eine OpenAI-kompatible API für mehrere Clients bereit. Der Server ist Teil des Ollama-Ökosystems und ermöglicht es, Modelle zentral zu hosten und über standardisierte Schnittstellen anzusprechen.

Im Unterschied zu anderen Servern ist die Integration stärker auf lokale und einfach deploybare Setups ausgelegt, wobei typischerweise mehrere Modelle innerhalb derselben Laufzeitumgebung verwaltet werden können.

Die Verbindung erfolgt über den API-Endpoint des Servers. Der verwendete API-Key dient dabei lediglich als Platzhalter (je nach Konfiguration wird bei Ollama kein echter API-Key benötigt).

Nach dem Setzen der Umgebungsvariablen deployen wir den Inference-Server mittels Helm.

In [ ]:
%%bash
source ~/work/env.py
cat <<EOF | tee ~/work/env.py
OPENAI_API_KEY="$OPENAI_API_KEY"
HF_TOKEN=""
AI_KUBECONFIG="$AI_KUBECONFIG"
AI_MODEL="llama3.1:8b-instruct-q4_K_M"
AI_NAME="ollama"
AI_IP="${AI_IP}"
EOF

Als nächstes ist das helm Chart zu konfigurieren, dass es die GPU verwendet.

In [ ]:
%%bash
source ~/work/env.py
cat <<EOF | tee values.yaml
ollama:
  gpu:
    enabled: true
    type: nvidia
    number: 1

  resources:
    limits:
      nvidia.com/gpu: 1
    requests:
      nvidia.com/gpu: 1

  models:
    pull:
      - "$AI_MODEL"

runtimeClassName: nvidia
EOF


In [ ]:
%%bash
source ~/work/env.py
helm ${AI_KUBECONFIG} repo add ollama-helm https://otwld.github.io/ollama-helm/
helm ${AI_KUBECONFIG} repo update
helm ${AI_KUBECONFIG} upgrade --install ${AI_NAME} --create-namespace ollama-helm/ollama --namespace ${AI_NAME} -f values.yaml

Bevor wir den einfachen Test starten können, muss der Inferenz Server bereit sein:

In [ ]:
%%bash
source ~/work/env.py
kubectl ${AI_KUBECONFIG} get pod,service -n ${AI_NAME} -l app.kubernetes.io/name=${AI_NAME}
kubectl ${AI_KUBECONFIG} logs -l app.kubernetes.io/name=${AI_NAME} --tail=10 -n ${AI_NAME} || true

Nach dem Start sollte ein Model sichtbar sein

In [ ]:
%%bash
source ~/work/env.py
POD=$(kubectl ${AI_KUBECONFIG} get pods -n ${AI_NAME} -l app.kubernetes.io/name=${AI_NAME} -o jsonpath="{.items[0].metadata.name}")
kubectl ${AI_KUBECONFIG} exec -n ${AI_NAME} $POD -- ollama list

Weitere Modelle können wie folgt geladen werden

In [ ]:
%%bash
source ~/work/env.py
POD=$(kubectl ${AI_KUBECONFIG} get pods -n ${AI_NAME} -l app.kubernetes.io/name=${AI_NAME} -o jsonpath="{.items[0].metadata.name}")
kubectl ${AI_KUBECONFIG} exec -n ${AI_NAME} $POD -- ollama pull 'qwen2.5:7b-instruct-q4_K_M'
kubectl ${AI_KUBECONFIG} exec -n ${AI_NAME} $POD -- ollama list

### Testen

Vorher muss base_url gesetzt werden.

In [ ]:
%%bash
source ~/work/env.py
kubectl ${AI_KUBECONFIG} patch svc "${AI_NAME}" -n "${AI_NAME}" -p '{"spec":{"type":"NodePort"}}'
echo "AI_BASE_URL=\"http://${AI_IP}:$(kubectl $AI_KUBECONFIG get service ${AI_NAME} -n ${AI_NAME} -o jsonpath='{.spec.ports[0].nodePort}')/v1\"" | tee -a ~/work/env.py

In [ ]:
%run ~/work/env.py
from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY, base_url=AI_BASE_URL)

response = client.responses.create(
    model=AI_MODEL,
    input="Wer war John F. Kennedy?"
)

print(response.output_text)

---
### Weitere Beispiele

Wechselt in das Verzeichnis `03-openai` und spielt die Übungen durch. 

Welche Funktionieren und welche nicht?

---
### Aufräumen

In [ ]:
%%bash
source ~/work/env.py
helm ${AI_KUBECONFIG} uninstall ${AI_NAME} -n ${AI_NAME} || true
kubectl ${AI_KUBECONFIG} delete ns ${AI_NAME} || true